# **Child Well-being - Data Partitioner**<br/>
**University: University of Milano-Bicocca**<br/>
**Master's Degree: Data Science (A.Y. 2025/2026)**<br/>
**Course: Data Science Lab**<br/>

In [25]:
import polars as pl

In [26]:
df = pl.read_parquet('../data/030_child_well_being.parquet')

In [27]:
# split data into 2 dataframes for each TIME_PERIOD
# - the firstone should contains public expenditure indicators + REF_AREA
# - the ladder should contains all the other indicators (outcomes and policies/resources indicators) + REF_AREA

public_expenditure_columns = [
    "C2_1", "C2_2", "C2_3",
    "C3_1", 
    "C4_4", "C4_5", 
    "C5_1"
]

indicator_columns = [col for col in df.columns if col not in public_expenditure_columns and col not in ["REF_AREA", "TIME_PERIOD"]]

time_periods = df.select("TIME_PERIOD").unique().to_series().to_list()

for time_period in time_periods:
    df_time_period = df.filter(pl.col("TIME_PERIOD") == time_period)
    df_public_expenditure = df_time_period.select(["TIME_PERIOD", "REF_AREA"] + public_expenditure_columns)
    df_indicators = df_time_period.select(["TIME_PERIOD", "REF_AREA"] + indicator_columns)

    df_public_expenditure.write_parquet(f'../data/040_public_expenditure_{time_period}.parquet')
    df_indicators.write_parquet(f'../data/040_indicators_{time_period}.parquet')
